# 原始 Stage1 39 特征 + CFTC 纳指 Mini 持仓变化 H5 信息门

冻结计划先运行无拟合 Qlib DatasetH 与成熟时间预检；2023–2026 为复用开发历史。

In [1]:
import os,json,hashlib,sys,sklearn
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
root=Path.cwd();card=root/'research/stage1_cftc_nq_positioning_h5_information_v2_20261023';plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized_by_active_Goal'] and sys.version_info[:2]==(3,12) and sklearn.__version__=='1.9.1'
parent=plan['parent'];parent_card=root/parent['source_card']
for name,key in [('inputs/features','feature_artifact_sha256'),('inputs/daily_frame','frame_artifact_sha256'),('inputs/timing','timing_artifact_sha256'),('models/base_outer_71','base_model_artifact_sha256')]:assert sha(parent_card/'qlib_recorder'/parent['experiment_id']/parent['recorder_id']/'artifacts'/name)==parent[key]
assert json.loads((root/parent['nas_readback']).read_text())['NAS_archive_sha256']==parent['nas_archive_sha256']
spec=plan['source'];assert sha(root/spec['summary'])==spec['summary_sha256'];source_summary=json.loads((root/spec['summary']).read_text());assert source_summary['recorder_id']==spec['recorder_id'] and source_summary['nas_readback']['verified']
source_artifact=Path(source_summary['recorder_uri'])/source_summary['experiment_id']/source_summary['recorder_id']/'artifacts'/spec['feature_artifact'];assert sha(source_artifact)==spec['mapped_artifact_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.uri_context(str(parent_card/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=parent['recorder_id'],experiment_id=parent['experiment_id']);original=rec.load_object('models/base_outer_71');base=rec.load_object('inputs/features');frame=rec.load_object('inputs/daily_frame');timing=rec.load_object('inputs/timing')
base_names=list(original.base_features);assert len(base_names)==39 and set(base_names).issubset(base.columns)
assert frame.index.equals(base.index) and frame.index.equals(timing.index)
with R.uri_context(source_summary['recorder_uri']):mapped=R.get_recorder(recorder_id=source_summary['recorder_id'],experiment_id=source_summary['experiment_id']).load_object(spec['feature_artifact'])
assert len(mapped)==3964 and mapped.index.is_unique and int(mapped.loc['2011-01-03':'2022-12-30','source_known'].sum())==2947 and int(mapped.loc['2023-01-03':'2026-10-07','source_known'].sum())==915
feature=mapped['CFTC_NQ_LevNetOI_delta4'].astype(float).reindex(frame.index).rename(plan['single_mechanism']['feature'])
assert feature.notna().equals(mapped.source_known.reindex(frame.index).fillna(False))
forward=frame.soxx_forward_open_return.astype(float);parts=[forward.shift(-i) for i in range(5)];valid=np.logical_and.reduce([np.isfinite(x.to_numpy(float))&(x.to_numpy(float)>-1) for x in parts]);label=pd.Series(np.log1p(pd.concat(parts,axis=1)).sum(axis=1).where(valid),index=frame.index,name='soxx_forward_five_open_log_return')
mature=pd.to_datetime(timing.return_available_at.shift(-4),utc=True);cutoff=pd.Timestamp('2023-01-01',tz='America/New_York').tz_convert('UTC');last=pd.Timestamp('2026-10-10',tz='America/New_York').tz_convert('UTC')
train_dates=(frame.index>=pd.Timestamp('2011-01-03'))&(frame.index<=pd.Timestamp('2022-12-30'))&(mature<cutoff)&label.notna();held_dates=(frame.index>=pd.Timestamp('2023-01-03'))&(frame.index<=pd.Timestamp('2026-10-07'))&(mature<last)&label.notna()
assert train_dates.sum()==plan['cohorts']['expected_mature_train_rows'] and held_dates.sum()==plan['cohorts']['expected_mature_held_rows']
augmented=base[base_names].copy();augmented[feature.name]=feature
meta=pd.DataFrame({'label_available_at':mature,'train_eligible':train_dates,'held_eligible':held_dates,'source_known':mapped.source_known.reindex(frame.index).fillna(False)},index=frame.index)
static=pd.concat({'feature':augmented,'label':label.to_frame(),'meta':meta},axis=1);static.index=pd.MultiIndex.from_arrays([frame.index,['SOXL_SOXX_CASH']*len(frame)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['cohorts']['train_signal']),'held':tuple(plan['cohorts']['held_reused_diagnostic'])});train_prepared=dataset.prepare('train',col_set=['feature','label','meta']);held_prepared=dataset.prepare('held',col_set=['feature','label','meta']);assert len(train_prepared)>=train_dates.sum() and len(held_prepared)>=held_dates.sum()
report={'status':'Stage1_CFTC_NQ_delta4_DatasetH_label_maturity_preflight_passed_conditional_source_no_fit','plan_sha256':sha(card/'plan.json'),'parent_recorder_id':parent['recorder_id'],'source_recorder_id':source_summary['recorder_id'],'original_primary_features':len(base_names),'candidate_coordinates':1,'training_mature_rows':int(train_dates.sum()),'held_mature_rows':int(held_dates.sum()),'train_segment_rows':len(train_prepared),'held_segment_rows':len(held_prepared),'candidate_train_unknown_rows':int(feature.loc[train_dates].isna().sum()),'candidate_held_unknown_rows':int(feature.loc[held_dates].isna().sum()),'source_Recorder_object_loads':5,'DatasetH_builds':1,'new_model_fits':0,'new_native_accounts':0,'held_scores_computed':False,'historical_independence':False,'historical_first_public_qualified':False}
(card/'preflight_result.json').write_text(json.dumps(report,indent=2)+'\n');print({k:report[k] for k in ('status','training_mature_rows','held_mature_rows','candidate_train_unknown_rows','candidate_held_unknown_rows')})


## 冻结拟合与成对信息检验

只有独立设计审查与预检哈希匹配后执行。

In [2]:
import shlex,tarfile,tempfile,subprocess
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.nas_market_data import archive_nas_recorder,_settings
review_path=card/'design_review_final.json';assert review_path.exists()
review=json.loads(review_path.read_text());notebook=json.loads((card/(card.name+'.ipynb')).read_text());preflight_source=''.join(notebook['cells'][1]['source']);preflight_hash=hashlib.sha256(preflight_source.encode()).hexdigest();fit_source=''.join(notebook['cells'][3]['source']);fit_hash=hashlib.sha256(fit_source.encode()).hexdigest()
assert review['verdict']=='APPROVE_DESIGN' and review['plan_sha256']==sha(card/'plan.json') and review['preflight_sha256']==sha(card/'preflight_result.json') and review['preflight_cell_source_sha256']==preflight_hash and review['fit_cell_source_sha256']==fit_hash
assert report==json.loads((card/'preflight_result.json').read_text()) and not report['historical_first_public_qualified']
state_path=card/'run_state.json';tracking=card/'qlib_recorder';assert not state_path.exists() and not tracking.exists()
state={'plan_sha256':sha(card/'plan.json'),'fit_cell_source_sha256':fit_hash,'fit_attempts':[],'fits_done':[],'prediction_attempts':[],'predictions_done':[],'bootstrap_attempted':False,'new_native_accounts':0}
state_path.write_text(json.dumps(state,indent=2)+'\n')
tr=train_prepared.loc[train_prepared[('meta','train_eligible')].astype(bool)];he=held_prepared.loc[held_prepared[('meta','held_eligible')].astype(bool)]
assert len(tr)==report['training_mature_rows']==3014 and len(he)==report['held_mature_rows']==940
yname='soxx_forward_five_open_log_return';feature_name=plan['single_mechanism']['feature'];ytr=tr[('label',yname)].to_numpy(float);yhe=he[('label',yname)].to_numpy(float)
assert np.isfinite(ytr).all() and np.isfinite(yhe).all()
inputs={'baseline':(tr['feature'][base_names].to_numpy(float),he['feature'][base_names].to_numpy(float)),'candidate':(tr['feature'][base_names+[feature_name]].to_numpy(float),he['feature'][base_names+[feature_name]].to_numpy(float))}
assert np.isfinite(inputs['candidate'][0][:,-1]).sum()==len(inputs['candidate'][0])-report['candidate_train_unknown_rows'] and np.isfinite(inputs['candidate'][1][:,-1]).sum()==len(inputs['candidate'][1])-report['candidate_held_unknown_rows']
models={};predictions={}
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 recorder=R.get_recorder();state.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id));state_path.write_text(json.dumps(state,indent=2)+'\n')
 dataset.config(dump_all=True,recursive=True)
 recorder.save_objects(artifact_path='inputs',plan=plan,preflight=report,dataset=dataset,derived_coordinate=feature,source_hashes={'CFTC_NQ_positioning':spec['summary_sha256'],'original39':parent['feature_artifact_sha256']})
 assert len(recorder.load_object('inputs/dataset').prepare('train',col_set=['feature','label','meta']))==report['train_segment_rows']
 for family in ('baseline','candidate'):
  assert family not in state['fit_attempts'] and len(state['fit_attempts'])<plan['budget']['model_fit_attempts_max']
  state['fit_attempts'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0));model.fit(inputs[family][0],ytr)
  recorder.save_objects(artifact_path='models',**{family:model});models[family]=model;state['fits_done'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  state['prediction_attempts'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  prediction=model.predict(inputs[family][1]);assert np.isfinite(prediction).all()
  recorder.save_objects(artifact_path='predictions',**{family:pd.DataFrame({'signal_date':he.index.get_level_values('datetime'),'prediction':prediction,'target':yhe})});predictions[family]=prediction
  state['predictions_done'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n');recorder.log_metrics(completed_fits=len(state['fits_done']),completed_predictions=len(state['predictions_done']))
 assert len(state['fits_done'])==len(state['predictions_done'])==2
 dates=pd.DatetimeIndex(he.index.get_level_values('datetime'));daily=pd.DataFrame({'signal_date':dates,'target':yhe,'baseline_prediction':predictions['baseline'],'candidate_prediction':predictions['candidate']})
 daily['baseline_squared_error']=(daily.target-daily.baseline_prediction)**2;daily['candidate_squared_error']=(daily.target-daily.candidate_prediction)**2;daily['paired_mse_improvement']=daily.baseline_squared_error-daily.candidate_squared_error
 assert daily.signal_date.is_monotonic_increasing and len(daily)==940
 halves=[daily.loc[daily.signal_date.lt(pd.Timestamp('2025-01-01'))],daily.loc[daily.signal_date.ge(pd.Timestamp('2025-01-01'))]];assert min(map(len,halves))>300
 held_calendar=pd.DatetimeIndex(held_prepared.index.get_level_values('datetime'));assert len(held_calendar)==report['held_segment_rows']==944 and held_calendar.is_unique and held_calendar.is_monotonic_increasing
 full_calendar=pd.DataFrame({'signal_date':held_calendar}).merge(daily[['signal_date','paired_mse_improvement']],on='signal_date',how='left',validate='one_to_one')
 assert len(full_calendar)==944 and full_calendar.paired_mse_improvement.notna().sum()==940 and full_calendar.paired_mse_improvement.isna().sum()==4
 assert full_calendar.paired_mse_improvement.iloc[:-4].notna().all() and full_calendar.paired_mse_improvement.iloc[-4:].isna().all()
 bootstrap=plan['information_gate']['paired_stationary_bootstrap'];state['bootstrap_attempted']=True;state_path.write_text(json.dumps(state,indent=2)+'\n')
 interval,_,replicates=full_calendar_bootstrap(full_calendar.paired_mse_improvement.to_numpy(float),mean_block=bootstrap['mean_block_sessions'],resamples=bootstrap['draws'],seed=bootstrap['seed'],alpha=bootstrap['one_sided_alpha'])
 assert interval['calendar_rows']==944 and interval['scored_rows']==940
 coefficient=float(models['candidate'].named_steps['ridge'].coef_[-1])
 metrics={'baseline_held_mse':float(daily.baseline_squared_error.mean()),'candidate_held_mse':float(daily.candidate_squared_error.mean()),'held_paired_mse_improvement':float(daily.paired_mse_improvement.mean()),'held_half_paired_improvements':[float(x.paired_mse_improvement.mean()) for x in halves],'candidate_standardized_coordinate_coefficient':coefficient,'bootstrap_lower':interval['lower_bound'],'bootstrap_upper':interval['upper_bound'],'bootstrap':interval}
 gates={'coefficient_positive':coefficient>0,'full_held_mean_positive':metrics['held_paired_mse_improvement']>0,'both_halves_positive':all(x>0 for x in metrics['held_half_paired_improvements']),'bootstrap_lower_positive':metrics['bootstrap_lower']>0};passed=bool(all(gates.values()))
 daily['quarter']=daily.signal_date.dt.to_period('Q').astype(str);quarters=[{'quarter':q,'sessions':len(g),'paired_mean_mse_improvement':float(g.paired_mse_improvement.mean())} for q,g in daily.groupby('quarter',sort=True)]
 recorder.save_objects(artifact_path='analysis/daily',held=daily,full_calendar=full_calendar);recorder.save_objects(artifact_path='analysis',metrics=metrics,gates=gates,quarters=quarters,bootstrap_replicates=replicates,information_gate_pass=passed)
 recorder.log_metrics(held_paired_mse_improvement=metrics['held_paired_mse_improvement'],bootstrap_lower=metrics['bootstrap_lower'],information_gate_pass=int(passed))
summary={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'decision':'conditional_development_CFTC_NQ_information_pass_first_public_pending_no_account' if passed else 'not_promoted_fixed_CFTC_NQ_delta4_H5_information_failed','objective_achieved':False,'information_gate_pass':passed,'training_mature_rows':len(tr),'held_mature_rows':len(he),'metrics':metrics,'gates':gates,'quarterly_information_attribution':quarters,'new_model_fits':2,'new_bootstraps':1,'new_native_accounts':0,'historical_first_public_qualified':False,'reused_market_history':True,'independent_forward_qualified_days':0,'recorder_id':state['recorder_id'],'experiment_id':state['experiment_id'],'recorder_uri':str(tracking),'parent_feature_sha256':parent['feature_artifact_sha256'],'source_summary_sha256':spec['summary_sha256'],'artifact_policy':plan['artifact_policy'],'limitations':['CFTC annual compressed values lack complete first-public revision and release-clock certification','Information MSE is not fee-net SOXX funded account result','Reused held 2023-2026 is development feedback, not independent forward']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');(card/'result_review.json').write_text(json.dumps({'status':summary['decision'],'plan_sha256':summary['plan_sha256'],'recorder_id':state['recorder_id'],'metrics':metrics,'gates':gates,'new_model_fits':2,'new_native_accounts':0},indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=state['recorder_id'],experiment_id=state['experiment_id'])
with tempfile.TemporaryDirectory(prefix='cftc-nq-info-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:names=[m.name for m in source.getmembers() if m.isfile()]
 assert any(name.endswith('/artifacts/models/baseline') for name in names) and any(name.endswith('/artifacts/models/candidate') for name in names)
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':evidence['sha256'],'verified':True};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['completed']=True;state['nas_archive_sha256']=evidence['sha256'];state_path.write_text(json.dumps(state,indent=2)+'\n')
print({k:summary[k] for k in ('decision','information_gate_pass','new_model_fits','new_native_accounts','recorder_id')});print(metrics)


{'decision': 'not_promoted_fixed_CFTC_NQ_delta4_H5_information_failed', 'information_gate_pass': False, 'new_model_fits': 2, 'new_native_accounts': 0, 'recorder_id': 'c2a9af571dbf486fb7b9ba279e0b9571'}
{'baseline_held_mse': 0.0025262669365982205, 'candidate_held_mse': 0.002543961155931939, 'held_paired_mse_improvement': -1.769421933371854e-05, 'held_half_paired_improvements': [6.321699327697393e-07, -3.869843716882611e-05], 'candidate_standardized_coordinate_coefficient': 0.0015153548068414493, 'bootstrap_lower': -6.441077928794541e-05, 'bootstrap_upper': 2.057971519741317e-05, 'bootstrap': {'paired_mean_loss_improvement': -1.769421933371854e-05, 'lower_bound': -6.441077928794541e-05, 'upper_bound': 2.057971519741317e-05, 'valid_replicates': 10000, 'invalid_replicates': 0, 'calendar_rows': 944, 'scored_rows': 940}}


## 中文结论

固定 CFTC 四周持仓变化 H5 信息门未通过：940 个成熟复用评价日，基线 MSE 0.002526267，候选 0.002543961；整体改善 -1.7694e-5，第二半 -3.8698e-5，校正下界 -6.4411e-5。系数符号虽为正，其余三门失败。两次拟合、一次 944 日历 bootstrap、零账户；Recorder 和 NAS 回读通过，独立审查确认不晋升。2026Q3 单季改善只作归因，详见 `conclusion.md`。